# 🔧 Step 2: Data Preprocessing (Continuous Time)

## What This Notebook Does

Takes the raw extracted data from Step 1 and transforms it into a single, clean,
analysis-ready dataset: `sepsis_continuous.csv`.

### Key Design Decision: Continuous Time (Not Hourly Grid)

**Previous approach** (common in literature): Resample all measurements to fixed 1-hour intervals.
This loses information — a measurement taken at 2:17 PM and another at 2:43 PM would be merged.

**Our approach**: Keep the **original measurement timestamps**. Each row in the output represents
one actual measurement event at its exact time.

This means:
- `hours_in` is a **float** (e.g., 2.283, 2.717) not an integer (2, 3)
- The time between consecutive observations varies (15 min to several hours)
- We add `delta_t` = time gap since previous observation as an explicit feature
- The HMM learns that a 6-hour gap carries different information than a 30-minute gap

### Processing Steps
1. Load all raw data files from Step 1
2. Convert timestamps to `hours_in` (hours since ICU admission)
3. Remove outliers and standardize units (e.g., Fahrenheit → Celsius)
4. Merge vitals + labs onto a unified continuous timeline per patient
5. Compute treatment flags (was the patient on antibiotics/vasopressors/fluids at each time point?)
6. Forward fill missing values (with time limits for vitals)
7. Compute `delta_t` and add patient-level features
8. Save as `sepsis_continuous.csv`

## 📌 Cell 1: Configuration

In [1]:
import pandas as pd
import numpy as np
import os
import time
import warnings
warnings.filterwarnings('ignore')

# =============================================================
# 🔧 USER CONFIGURATION
# =============================================================
PROCESSED_DIR = "/Users/hoon/Documents/10_Classes/12_Bayesian Machine Learning with Generative AI Applications/10_Final_Project/02_Processed_Data"
OUTPUT_DIR = "/Users/hoon/Documents/10_Classes/12_Bayesian Machine Learning with Generative AI Applications/10_Final_Project/03_Model_Ready_Data"

# =============================================================
# 🔧 TUNABLE: Forward fill limits
# =============================================================
VITAL_FFILL_HOURS = 2     # Vitals: max 2h forward fill
LAB_FFILL_HOURS = None     # Lab: unlimited forward fill (tests are infrequent)

os.makedirs(OUTPUT_DIR, exist_ok=True)
start_time = time.time()

## 📌 Cell 2: Load Raw Data

Load all files from Step 1's output directory. Also prepare key mappings:
- Convert datetime strings to pandas datetime objects
- Identify valid patients (those in the cohort)
- Set up the ICU time windows for each patient

In [2]:
print("=" * 60)
print("PART A: Data loading")
print("=" * 60)

cohort = pd.read_csv(f"{PROCESSED_DIR}/cohort.csv")
vitals = pd.read_csv(f"{PROCESSED_DIR}/vitals.csv")
labs = pd.read_csv(f"{PROCESSED_DIR}/labs.csv")
inputevents = pd.read_csv(f"{PROCESSED_DIR}/inputevents.csv")
admissions = pd.read_csv(f"{PROCESSED_DIR}/admissions.csv")
patients = pd.read_csv(f"{PROCESSED_DIR}/patients.csv")

# Load outputevents (if exists)
try:
    outputevents = pd.read_csv(f"{PROCESSED_DIR}/outputevents.csv")
except:
    outputevents = pd.DataFrame()

# Load prescriptions (if exists)
try:
    prescriptions = pd.read_csv(f"{PROCESSED_DIR}/prescriptions.csv")
except:
    prescriptions = pd.DataFrame()

for name, df in [('cohort', cohort), ('vitals', vitals), ('labs', labs),
                  ('inputevents', inputevents), ('admissions', admissions)]:
    print(f"  {name:15s}: {len(df):>12,}rows")

# Time conversion
cohort['intime'] = pd.to_datetime(cohort['intime'])
cohort['outtime'] = pd.to_datetime(cohort['outtime'])

valid_stay_ids = set(cohort['stay_id'].unique())
valid_hadm_ids = set(cohort['hadm_id'].unique())
valid_subject_ids = set(cohort['subject_id'].unique())
icu_intime_lookup = cohort.set_index('stay_id')['intime'].to_dict()
icu_outtime_lookup = cohort.set_index('stay_id')['outtime'].to_dict()

# Mortality info
admissions['deathtime'] = pd.to_datetime(admissions['deathtime'])
cohort = cohort.merge(
    admissions[['hadm_id', 'hospital_expire_flag', 'race', 'admission_type']].drop_duplicates(),
    on='hadm_id', how='left'
)
cohort['mortality'] = cohort['hospital_expire_flag'].fillna(0).astype(int)
cohort['icu_los_hours'] = (cohort['outtime'] - cohort['intime']).dt.total_seconds() / 3600

print(f"\n  Cohort: {len(cohort):,} ICU stays")
print(f"  Mortality rate: {cohort['mortality'].mean():.1%}")

PART A: Data loading
  cohort         :        1,502rows
  vitals         :    1,301,329rows
  labs           :    1,052,508rows
  inputevents    :      331,093rows
  admissions     :        1,502rows

  Cohort: 1,502 ICU stays
  Mortality rate: 17.2%


## 📌 Cell 3: Vital Sign Cleaning (Continuous Time)

For each vital sign measurement:
1. **Map item IDs to variable names** (e.g., 220045 → "heart_rate")
2. **Convert units**: Temperature Fahrenheit → Celsius
3. **Remove physiological outliers**: Values outside plausible ranges
   - Heart rate: 20-300 bpm (outside = measurement error)
   - Temperature: 32-42°C
   - SpO2: 50-100%
4. **Compute `hours_in`**: `(charttime - intime).total_seconds() / 3600`
5. **Filter to ICU window**: Only keep measurements during the ICU stay
6. **Pivot to wide format**: One column per vital sign

In [3]:
print("\n" + "=" * 60)
print("PART B: Vital sign cleaning (Continuous Time)")
print("=" * 60)

vitals = vitals[vitals['stay_id'].isin(valid_stay_ids)].copy()
vitals['charttime'] = pd.to_datetime(vitals['charttime'])

# B1: itemid → unified variables
VITAL_NAME_MAP = {
    220045: 'heart_rate',
    220050: 'sbp', 225312: 'sbp', 220179: 'sbp',
    220051: 'dbp', 220180: 'dbp',
    220052: 'map', 220181: 'map',
    223761: 'temp_f', 223762: 'temp_c',
    220277: 'spo2', 220235: 'fio2',
    220210: 'resp_rate', 224690: 'resp_rate',
    220739: 'gcs_eye', 223900: 'gcs_verbal', 223901: 'gcs_motor',
    226559: 'urine_output',
}

vitals['vital_name'] = vitals['itemid'].map(VITAL_NAME_MAP)
vitals = vitals.dropna(subset=['vital_name', 'valuenum'])

# B2: Fahrenheit → Celsius conversion
temp_f_mask = vitals['vital_name'] == 'temp_f'
vitals.loc[temp_f_mask, 'valuenum'] = (vitals.loc[temp_f_mask, 'valuenum'] - 32) * 5 / 9
vitals.loc[temp_f_mask, 'vital_name'] = 'temperature'
vitals.loc[vitals['vital_name'] == 'temp_c', 'vital_name'] = 'temperature'

# B3: Outlier removal
VITAL_RANGES = {
    'heart_rate':   (20, 300),
    'sbp':          (30, 300),
    'dbp':          (10, 200),
    'map':          (20, 250),
    'temperature':  (30, 45),
    'spo2':         (50, 100),
    'fio2':         (21, 100),
    'resp_rate':    (4, 60),
    'gcs_eye':      (1, 4),
    'gcs_verbal':   (1, 5),
    'gcs_motor':    (1, 6),
    'urine_output': (0, 2500),
}

rows_before = len(vitals)
for vital, (vmin, vmax) in VITAL_RANGES.items():
    mask = vitals['vital_name'] == vital
    vitals = vitals[~(mask & ((vitals['valuenum'] < vmin) | (vitals['valuenum'] > vmax)))]
rows_removed = rows_before - len(vitals)
print(f"  Outlier removal: {rows_removed:,}rows ({rows_removed/rows_before*100:.1f}%)")

# B4: Relative time (continuous — hours_in is float!)
vitals['icu_intime'] = vitals['stay_id'].map(icu_intime_lookup)
vitals['hours_in'] = (vitals['charttime'] - vitals['icu_intime']).dt.total_seconds() / 3600

# Only within ICU stay period
vitals = vitals[(vitals['hours_in'] >= 0)].copy()
# Remove after each stay outtime
vitals['icu_outtime'] = vitals['stay_id'].map(icu_outtime_lookup)
vitals = vitals[vitals['charttime'] <= vitals['icu_outtime']].copy()
vitals = vitals.drop(columns=['icu_intime', 'icu_outtime'])

# B5: Wide format (average duplicate vitals at same time)
vitals_wide = vitals.pivot_table(
    index=['stay_id', 'hours_in', 'charttime'],
    columns='vital_name',
    values='valuenum', aggfunc='mean'
).reset_index()
vitals_wide.columns.name = None

print(f"  Cleaning complete: {len(vitals_wide):,} time points (continuous)")
print(f"  stays: {vitals_wide['stay_id'].nunique():,}")


PART B: Vital sign cleaning (Continuous Time)
  Outlier removal: 1,793rows (0.1%)
  Cleaning complete: 265,697 time points (continuous)
  stays: 1,502


## 📌 Cell 4: Lab Result Cleaning (Continuous Time)

Similar cleaning process for lab values. Key differences:
- Labs have **no `stay_id`** — must link via `subject_id` and time
- Labs are measured less frequently (every 4-12 hours vs every 15-60 min for vitals)
- Outlier ranges are different for each lab (e.g., Lactate > 30 is impossible)

In [4]:
print("\n" + "=" * 60)
print("PART C: Lab result cleaning (Continuous Time)")
print("=" * 60)

labs = labs[labs['subject_id'].isin(valid_subject_ids)].copy()
labs['charttime'] = pd.to_datetime(labs['charttime'])

LAB_NAME_MAP = {
    50813: 'lactate', 51652: 'procalcitonin', 50889: 'crp',
    51265: 'platelets', 50885: 'bilirubin', 50912: 'creatinine',
    51301: 'wbc', 51300: 'wbc', 51222: 'hemoglobin', 51221: 'hematocrit',
    51006: 'bun', 50862: 'albumin', 50931: 'glucose',
    50971: 'potassium', 50983: 'sodium', 50902: 'chloride',
    50820: 'ph', 50821: 'po2', 50818: 'pco2',
    50882: 'bicarbonate', 50802: 'base_excess', 51237: 'inr',
}

labs['lab_name'] = labs['itemid'].map(LAB_NAME_MAP)
labs = labs.dropna(subset=['lab_name', 'valuenum'])

LAB_RANGES = {
    'lactate': (0.1, 30), 'procalcitonin': (0, 200), 'crp': (0, 500),
    'platelets': (5, 1500), 'bilirubin': (0, 60), 'creatinine': (0.1, 25),
    'wbc': (0.1, 200), 'hemoglobin': (2, 25), 'hematocrit': (10, 70),
    'bun': (1, 200), 'albumin': (0.5, 6), 'glucose': (10, 1500),
    'potassium': (1.5, 10), 'sodium': (110, 180), 'chloride': (70, 140),
    'ph': (6.5, 8.0), 'po2': (20, 600), 'pco2': (10, 120),
    'bicarbonate': (5, 50), 'base_excess': (-30, 30), 'inr': (0.5, 15),
}

rows_before = len(labs)
for lab, (lmin, lmax) in LAB_RANGES.items():
    mask = labs['lab_name'] == lab
    labs = labs[~(mask & ((labs['valuenum'] < lmin) | (labs['valuenum'] > lmax)))]
rows_removed = rows_before - len(labs)
print(f"  Outlier removal: {rows_removed:,}rows ({rows_removed/rows_before*100:.1f}%)")

# Lab → ICU stay linkage
from_cohort = cohort[['hadm_id', 'stay_id', 'intime', 'outtime']].copy()
labs = labs.merge(from_cohort, on='hadm_id', how='inner')
labs = labs[labs['stay_id'].isin(valid_stay_ids)]

# Relative time (continuous)
labs['hours_in'] = (labs['charttime'] - labs['intime']).dt.total_seconds() / 3600
labs = labs[(labs['hours_in'] >= 0)].copy()
labs = labs[labs['charttime'] <= labs['outtime']].copy()

labs_wide = labs.pivot_table(
    index=['stay_id', 'hours_in', 'charttime'],
    columns='lab_name',
    values='valuenum', aggfunc='mean'
).reset_index()
labs_wide.columns.name = None

print(f"  Cleaning complete: {len(labs_wide):,} time points (continuous)")


PART C: Lab result cleaning (Continuous Time)
  Outlier removal: 164rows (0.0%)
  Cleaning complete: 31,317 time points (continuous)


## 📌 Cell 5: Treatment Identification

Create binary treatment flags: At any given time, is the patient receiving...
- **Antibiotics** (on_antibiotics: 0 or 1)
- **Vasopressors** (on_vasopressors: 0 or 1)
- **IV Fluids** (on_iv_fluids: 0 or 1)

These are derived from inputevents (start/end times) and prescriptions (keyword matching).

In [5]:
print("\n" + "=" * 60)
print("PART D: Treatment identification")
print("=" * 60)

# Antibiotic keywords (prescriptions-based)
ANTIBIOTIC_KEYWORDS = [
    'vancomycin', 'piperacillin', 'tazobactam', 'meropenem',
    'imipenem', 'cefepime', 'ceftriaxone', 'ceftazidime',
    'ciprofloxacin', 'levofloxacin', 'moxifloxacin',
    'azithromycin', 'metronidazole', 'linezolid', 'daptomycin',
    'amoxicillin', 'ampicillin', 'nafcillin', 'oxacillin',
    'gentamicin', 'tobramycin', 'amikacin',
    'trimethoprim', 'sulfamethoxazole', 'doxycycline',
    'clindamycin', 'erythromycin', 'aztreonam',
    'cefazolin', 'cephalexin', 'cefuroxime', 'cefoxitin',
    'ertapenem', 'doripenem', 'colistin', 'polymyxin',
    'tigecycline', 'nitrofurantoin', 'fosfomycin',
    'fluconazole', 'micafungin', 'caspofungin', 'voriconazole', 'amphotericin',
]

# Vasopressor itemids (inputevents)
VASOPRESSOR_ITEMIDS = [
    221906, 221289, 222315, 221749, 221662, 229617, 221986,
    221653, 229630, 229631, 229632, 229709, 229764,
]

# IV fluid itemids
IV_FLUID_ITEMIDS = [
    220949, 225158, 225159, 225168, 220950, 225828,
    225944, 225797, 225825, 225827, 220952, 225161,
]

inputevents['starttime'] = pd.to_datetime(inputevents['starttime'])
inputevents['endtime'] = pd.to_datetime(inputevents['endtime'])

# Extract vasopressor/fluid events
vaso_events = inputevents[inputevents['itemid'].isin(VASOPRESSOR_ITEMIDS)].copy()
fluid_events = inputevents[inputevents['itemid'].isin(IV_FLUID_ITEMIDS)].copy()
print(f"  Vasopressor events: {len(vaso_events):,}")
print(f"  Fluid events:   {len(fluid_events):,}")

# Antibiotics (prescriptions-based)
if not prescriptions.empty:
    prescriptions['drug_lower'] = prescriptions['drug'].fillna('').str.lower()
    abx_mask = prescriptions['drug_lower'].apply(
        lambda x: any(kw in x for kw in ANTIBIOTIC_KEYWORDS))
    abx_prescriptions = prescriptions[abx_mask].copy()
    abx_prescriptions['starttime'] = pd.to_datetime(abx_prescriptions['starttime'])
    abx_prescriptions['stoptime'] = pd.to_datetime(abx_prescriptions['stoptime'])
    # ICU stay linkage
    abx_with_stay = abx_prescriptions.merge(
        cohort[['hadm_id', 'stay_id', 'intime']], on='hadm_id', how='inner')
    abx_with_stay['intime'] = pd.to_datetime(abx_with_stay['intime'])
    print(f"  Antibiotic prescriptions: {len(abx_with_stay):,}")
else:
    abx_with_stay = pd.DataFrame()
    print("  Antibiotic prescriptions: prescriptions.csv not found — using inputevents antibiotics")


PART D: Treatment identification
  Vasopressor events: 22,415
  Fluid events:   97,232
  Antibiotic prescriptions: 16,045


## 📌 Cell 6: Continuous Timeline Integration

**The most complex cell in this notebook.**

For each patient:
1. Collect ALL unique observation time points from both vitals and labs
2. Create a master timeline with these exact timestamps
3. Merge vital signs (by exact `hours_in` match)
4. Merge lab values (by exact `hours_in` match)
5. For each observation time, check if any treatment was active
   (i.e., does `hours_in` fall within any treatment's [start, end] interval?)
6. Forward fill missing values:
   - **Vitals**: Fill forward up to **2 hours** (vitals change quickly)
   - **Labs**: Fill forward with **no time limit** (labs are infrequent)
7. Compute **delta_t**: Time gap between consecutive observations

This preserves the natural, irregular sampling of ICU data.

In [6]:
print("\n" + "=" * 60)
print("PART E: Continuous Timeline Integration")
print("=" * 60)

# Build unified timeline per patient
all_patient_dfs = []

for sid in sorted(valid_stay_ids):
    intime = icu_intime_lookup[sid]
    outtime = icu_outtime_lookup[sid]
    
    # This patient vitals & labs time points
    v = vitals_wide[vitals_wide['stay_id'] == sid].copy()
    l = labs_wide[labs_wide['stay_id'] == sid].copy()
    
    if v.empty and l.empty:
        continue
    
    # Collect all unique time points
    all_times = set()
    if not v.empty:
        all_times.update(v['hours_in'].values)
    if not l.empty:
        all_times.update(l['hours_in'].values)
    
    all_times = sorted(all_times)
    
    # Master timeline generation
    master = pd.DataFrame({'hours_in': all_times})
    master['stay_id'] = sid
    
    # Merge vitals (exact match on hours_in)
    if not v.empty:
        vital_cols = [c for c in v.columns if c not in ['stay_id', 'hours_in', 'charttime']]
        v_merge = v[['hours_in'] + vital_cols].copy()
        # Average if multiple values at same time
        v_merge = v_merge.groupby('hours_in')[vital_cols].mean().reset_index()
        master = master.merge(v_merge, on='hours_in', how='left')
    
    # Merge labs
    if not l.empty:
        lab_cols = [c for c in l.columns if c not in ['stay_id', 'hours_in', 'charttime']]
        l_merge = l[['hours_in'] + lab_cols].copy()
        l_merge = l_merge.groupby('hours_in')[lab_cols].mean().reset_index()
        master = master.merge(l_merge, on='hours_in', how='left')
    
    # Sort by time
    master = master.sort_values('hours_in').reset_index(drop=True)
    
    # --- Treatment flags (continuous time) ---
    # Check if treatment active at each obs time
    master['on_vasopressors'] = 0
    master['on_antibiotics'] = 0
    master['on_iv_fluids'] = 0
    
    # Vasopressors
    vaso_sub = vaso_events[vaso_events['stay_id'] == sid]
    for _, ev in vaso_sub.iterrows():
        start_h = (ev['starttime'] - intime).total_seconds() / 3600
        end_h = (ev['endtime'] - intime).total_seconds() / 3600 if pd.notna(ev['endtime']) else start_h
        mask = (master['hours_in'] >= start_h) & (master['hours_in'] <= end_h)
        master.loc[mask, 'on_vasopressors'] = 1
    
    # IV fluids
    fluid_sub = fluid_events[fluid_events['stay_id'] == sid]
    for _, ev in fluid_sub.iterrows():
        start_h = (ev['starttime'] - intime).total_seconds() / 3600
        end_h = (ev['endtime'] - intime).total_seconds() / 3600 if pd.notna(ev['endtime']) else start_h
        mask = (master['hours_in'] >= start_h) & (master['hours_in'] <= end_h)
        master.loc[mask, 'on_iv_fluids'] = 1
    
    # Antibiotics
    if not abx_with_stay.empty:
        abx_sub = abx_with_stay[abx_with_stay['stay_id'] == sid]
        for _, ev in abx_sub.iterrows():
            start_h = (ev['starttime'] - intime).total_seconds() / 3600
            stop_t = ev['stoptime'] if pd.notna(ev['stoptime']) else ev['starttime']
            end_h = (stop_t - intime).total_seconds() / 3600
            mask = (master['hours_in'] >= start_h) & (master['hours_in'] <= end_h)
            master.loc[mask, 'on_antibiotics'] = 1
    
    # --- Forward fill (continuous time version) ---
    # Vitals: time-based ffill (within 2 hours)
    if 'heart_rate' in master.columns:
        vital_cols_available = [c for c in ['heart_rate', 'sbp', 'dbp', 'map',
                                             'temperature', 'spo2', 'fio2',
                                             'resp_rate'] if c in master.columns]
        for col in vital_cols_available:
            # forward fill with time limit
            last_val = np.nan
            last_time = -999
            vals = master[col].values.copy()
            times = master['hours_in'].values
            for i in range(len(vals)):
                if pd.notna(vals[i]):
                    last_val = vals[i]
                    last_time = times[i]
                elif pd.notna(last_val) and (times[i] - last_time) <= VITAL_FFILL_HOURS:
                    vals[i] = last_val
            master[col] = vals
    
    # Labs: forward fill (unlimited or per config)
    if 'lactate' in master.columns:
        lab_cols_available = [c for c in ['lactate', 'wbc', 'platelets', 'creatinine',
                                           'bilirubin', 'hemoglobin', 'albumin', 'glucose',
                                           'potassium', 'sodium', 'ph', 'po2', 'pco2',
                                           'bicarbonate', 'inr', 'procalcitonin', 'crp',
                                           'hematocrit', 'bun', 'chloride', 'base_excess'
                                           ] if c in master.columns]
        for col in lab_cols_available:
            if LAB_FFILL_HOURS is None:
                master[col] = master[col].ffill()
            else:
                last_val = np.nan
                last_time = -999
                vals = master[col].values.copy()
                times = master['hours_in'].values
                for i in range(len(vals)):
                    if pd.notna(vals[i]):
                        last_val = vals[i]
                        last_time = times[i]
                    elif pd.notna(last_val) and (times[i] - last_time) <= LAB_FFILL_HOURS:
                        vals[i] = last_val
                master[col] = vals
    
    # Compute delta_t (time gap from previous observation)
    master['delta_t'] = master['hours_in'].diff().fillna(0)
    
    all_patient_dfs.append(master)

print(f"  patients {len(all_patient_dfs):,}patients processing complete")

# Full merge
final = pd.concat(all_patient_dfs, ignore_index=True)
print(f"  Total observation time points: {len(final):,}")


PART E: Continuous Timeline Integration
  patients 1,502patients processing complete
  Total observation time points: 284,673


## 📌 Cell 7: Add Patient-Level Characteristics

Merge demographic data (age, gender, mortality, ICU type) from the cohort file.

In [7]:
print("\n" + "=" * 60)
print("PART F: Add patient-level characteristics")
print("=" * 60)

patient_features = cohort[['stay_id', 'subject_id', 'hadm_id',
    'gender', 'anchor_age', 'mortality', 'icu_los_hours']].copy()
patient_features['male'] = (patient_features['gender'] == 'M').astype(int)

final = final.merge(
    patient_features[['stay_id', 'subject_id', 'hadm_id', 'male',
                       'anchor_age', 'mortality', 'icu_los_hours']],
    on='stay_id', how='left')

print(f"  Final shape: {final.shape}")
print(f"  Stays: {final['stay_id'].nunique():,}")
print(f"  Columns: {list(final.columns)}")


PART F: Add patient-level characteristics
  Final shape: (284673, 44)
  Stays: 1,502
  Columns: ['hours_in', 'stay_id', 'dbp', 'fio2', 'gcs_eye', 'gcs_motor', 'gcs_verbal', 'heart_rate', 'map', 'resp_rate', 'sbp', 'spo2', 'temperature', 'albumin', 'base_excess', 'bicarbonate', 'bilirubin', 'bun', 'chloride', 'creatinine', 'crp', 'glucose', 'hematocrit', 'hemoglobin', 'inr', 'lactate', 'pco2', 'ph', 'platelets', 'po2', 'potassium', 'procalcitonin', 'sodium', 'wbc', 'on_vasopressors', 'on_antibiotics', 'on_iv_fluids', 'delta_t', 'subject_id', 'hadm_id', 'male', 'anchor_age', 'mortality', 'icu_los_hours']


## 📌 Cell 8: Save Final Dataset

Save `sepsis_continuous.csv` — the single clean file that all subsequent notebooks use.

In [8]:
print("\n" + "=" * 60)
print("PART G: Save")
print("=" * 60)

# ⚡ Key output: sepsis_continuous.csv (continuous, not hourly!)
final.to_csv(f"{OUTPUT_DIR}/sepsis_continuous.csv", index=False)
print(f"  sepsis_continuous.csv ({len(final):,}rows)")

cohort.to_csv(f"{OUTPUT_DIR}/cohort.csv", index=False)
print(f"  cohort.csv ({len(cohort):,}rows)")

total_time = time.time() - start_time
print(f"\n✅ Preprocessing complete! ({total_time/60:.1f}min)")
print(f"  Output: {OUTPUT_DIR}")
print("\n🔜 Next: 03_EDA.ipynb Run")


PART G: Save
  sepsis_continuous.csv (284,673rows)
  cohort.csv (1,502rows)

✅ Preprocessing complete! (0.7min)
  Output: /Users/hoon/Documents/10_Classes/12_Bayesian Machine Learning with Generative AI Applications/10_Final_Project/03_Model_Ready_Data

🔜 Next: 03_EDA.ipynb Run
